Poniżej masz wersję **w Markdown, ale z kodami w Pythonie**. Oparłem ją o bibliotekę **scikit-fda**, która udostępnia klasy `ConstantBasis`, `MonomialBasis`, `FourierBasis` i `BSplineBasis`, czyli bezpośrednie odpowiedniki baz znanych z R. Obiekt reprezentacji bazowej danych funkcjonalnych w Pythonie to `FDataBasis`. W dokumentacji scikit-fda baza Fouriera ma zawsze nieparzystą liczbę funkcji bazowych — jeśli podasz liczbę parzystą, zostanie automatycznie zwiększona o 1. Dokumentacja potwierdza też, że `BSplineBasis` używa parametrów `domain_range`, `n_basis`, `order` i `knots`, a obiekty baz mają metody `plot()` oraz obsługę pochodnych. ([fda.readthedocs.io][1])

---

# Funkcje tworzące bazy w Pythonie

W Pythonie, przy użyciu biblioteki `scikit-fda`, można definiować różne systemy bazowe funkcji. Odpowiedniki najważniejszych baz z R są następujące:

```python
from skfda.representation.basis import (
    ConstantBasis,
    MonomialBasis,
    FourierBasis,
    BSplineBasis,
)
```

Przykładowe tworzenie obiektów bazowych:

```python
basisobj = ConstantBasis(domain_range=(0, 1))
basisobj = MonomialBasis(domain_range=(0, 1), n_basis=4)
basisobj = FourierBasis(domain_range=(0, 1), n_basis=5, period=1)
basisobj = BSplineBasis(domain_range=(0, 1), n_basis=10, order=4)
```

Jeśli chcemy reprezentować konkretną funkcję lub zbiór funkcji w zadanej bazie, używamy obiektu `FDataBasis`. ([fda.readthedocs.io][2])

---

## Najczęściej używane argumenty

* `domain_range` – zakres argumentu ( t ), odpowiednik `rangeval` w R; najczęściej para `(a, b)`
* `n_basis` – liczba funkcji bazowych ( K )
* `order` – rząd splajnu B-spline
* `knots` – wektor węzłów
* `period` – okres bazy Fouriera

Przykład przedziału jednostkowego:

```python
domain_range = (0, 1)
```

---

# Baza Fouriera

## Przykład

Poniższy kod tworzy bazę Fouriera z ( K = 65 ) funkcjami bazowymi na przedziale ([0, 365]) i z okresem równym 365:

```python
from skfda.representation.basis import FourierBasis

daybasis65 = FourierBasis(domain_range=(0, 365), n_basis=65, period=365)
```

Jeśli nie podamy `period`, to jest on powiązany z zakresem dziedziny. Dodatkowo dokumentacja scikit-fda podaje, że liczba funkcji bazowych w bazie Fouriera jest zawsze nieparzysta; gdy podamy wartość parzystą, biblioteka automatycznie zwiększy ją o 1. ([fda.readthedocs.io][3])

---

## Rysowanie bazy Fouriera

```python
daybasis65.plot()
```

---

## Baza Fouriera bez wyrazu stałego

W R można było używać `dropind=1`. W Pythonie nie ma tu dokładnie tej samej składni, więc najprościej jest po prostu utworzyć bazę z odpowiednią liczbą funkcji i pamiętać, że w dalszych obliczeniach nie chcemy używać pierwszej funkcji bazowej, albo ręcznie zbudować własny układ funkcji. To jest jedna z różnic między `fda` w R a `scikit-fda` w Pythonie. Dokumentacja scikit-fda potwierdza klasy bazowe i ich parametry, ale nie opisuje odpowiednika `dropind` dla `FourierBasis`. ([fda.readthedocs.io][2])

---

# Zadanie 1 – B-splajny

## Utworzenie 13 B-splajnów rzędu czwartego na przedziale ([0,10])

```python
from skfda.representation.basis import BSplineBasis

splinebasis = BSplineBasis(domain_range=(0, 10), n_basis=13, order=4)
splinebasis.plot()
```

Dokumentacja `BSplineBasis` wskazuje, że `order=4` oznacza splajny stopnia trzeciego, czyli klasyczne splajny sześcienne. ([fda.readthedocs.io][4])

---

## Narysowanie pojedynczej funkcji bazowej

Tu trzeba uważać: w Pythonie nie zawsze zadziała indeksowanie dokładnie tak samo jak w R typu `basis[6]`. Najpewniejsza droga to ewaluacja wszystkich funkcji bazowych na siatce punktów i wybranie jednej kolumny do narysowania.

```python
import numpy as np
import matplotlib.pyplot as plt

grid = np.linspace(0, 10, 1000)
values = splinebasis(grid)[..., 0]   # wartości funkcji bazowych na siatce

plt.plot(grid, values[:, 5])  # 6. funkcja bazowa, bo indeksowanie od 0
plt.title("6. funkcja bazowa B-spline")
plt.xlabel("t")
plt.ylabel("wartość")
plt.show()
```

To podejście jest praktyczniejsze niż ślepe kopiowanie składni z R. Sama dokumentacja potwierdza, że obiekty baz można ewaluować i rysować. ([fda.readthedocs.io][5])

---

## Komentarz do własności

Dla B-splajnów rzędu czwartego funkcja bazowa jest niezerowa tylko lokalnie, na ograniczonej liczbie sąsiednich podprzedziałów. Wynika to z definicji rekurencyjnej B-splajnów i ich lokalnego nośnika. Dokumentacja scikit-fda opisuje tę definicję i sposób budowy na węzłach. ([fda.readthedocs.io][4])

---

# Zadanie 2 – różne rzędy splajnów

## Tworzenie baz

```python
import numpy as np
from skfda.representation.basis import BSplineBasis

basis2 = BSplineBasis(domain_range=(0, 2*np.pi), n_basis=5, order=2)
basis3 = BSplineBasis(domain_range=(0, 2*np.pi), n_basis=6, order=3)
basis4 = BSplineBasis(domain_range=(0, 2*np.pi), n_basis=7, order=4)
```

## Rysowanie

```python
basis2.plot()
basis3.plot()
basis4.plot()
```

---

## Baza z tymi samymi lokalizacjami węzłów, ale rzędu 6

Tu trzeba być ostrożnym: jeśli chcesz zachować dokładnie te same węzły, najlepiej jawnie je podać przez argument `knots`, zamiast liczyć, że sama liczba funkcji bazowych wszystko ustawi identycznie.

```python
knots = np.linspace(0, 2*np.pi, 4)  # przykład
basis6 = BSplineBasis(domain_range=(0, 2*np.pi), n_basis=9, order=6, knots=knots)
basis6.plot()
```

To jest ważne, bo przy B-splajnach relacja między `n_basis`, `order` i `knots` nie jest czymś, co warto zgadywać na pamięć — lepiej jawnie kontrolować węzły. Dokumentacja potwierdza, że `knots` można przekazać bezpośrednio. ([fda.readthedocs.io][4])

---

## Nierównomiernie rozmieszczone węzły

```python
knotvec = [0, 1, 2, 4, 7, 10]

basis_irregular = BSplineBasis(
    domain_range=(0, 10),
    n_basis=8,
    order=4,
    knots=knotvec
)

basis_irregular.plot()
```

---

# Zadanie 3 – inne bazy

## Baza stała na ([0,1])

```python
from skfda.representation.basis import ConstantBasis

conbasis = ConstantBasis(domain_range=(0, 1))
conbasis.plot()
```

## Baza jednomianowa stopnia 3 na ([0,1])

```python
from skfda.representation.basis import MonomialBasis

monbasis = MonomialBasis(domain_range=(0, 1), n_basis=4)
monbasis.plot()
```

Taka baza odpowiada funkcjom:
[
1,; t,; t^2,; t^3
]
co jest zgodne z ideą `MonomialBasis` w scikit-fda. ([fda.readthedocs.io][1])

---

# Zadanie 4 – wygenerowanie bazy B-splajnów

## Krok 1 – wybór zakresu

```python
domain_range = (0, 1)
```

## Krok 2 – wybór rzędu

```python
order = 4
```

## Krok 3 – liczba funkcji bazowych

```python
n_basis = 23
```

## Krok 4 – utworzenie bazy

```python
from skfda.representation.basis import BSplineBasis

basis = BSplineBasis(domain_range=domain_range, n_basis=n_basis, order=order)
```

## Krok 5 – rysowanie

```python
basis.plot()
```

---

## Pochodne bazy

W scikit-fda obiekty baz mają obsługę pochodnych. Dokumentacja bazowej klasy `Basis` podaje metody związane z pochodnymi, w tym `derivative`. ([fda.readthedocs.io][5])

Przykład praktyczny:

```python
basis_der1 = basis.derivative(order=1)
basis_der2 = basis.derivative(order=2)

basis_der1.plot()
basis_der2.plot()
```

---

## Alternatywnie: ręczne rysowanie pochodnych na siatce

Jeżeli chcesz mieć większą kontrolę nad wykresem:

```python
import numpy as np
import matplotlib.pyplot as plt

grid = np.linspace(0, 1, 1000)

vals0 = basis(grid)[..., 0]
vals1 = basis.derivative(order=1)(grid)[..., 0]
vals2 = basis.derivative(order=2)(grid)[..., 0]

plt.figure(figsize=(10, 4))
for i in range(vals0.shape[1]):
    plt.plot(grid, vals0[:, i])
plt.title("Baza B-splajnów")
plt.show()

plt.figure(figsize=(10, 4))
for i in range(vals1.shape[1]):
    plt.plot(grid, vals1[:, i])
plt.title("Pierwsze pochodne funkcji bazowych")
plt.show()

plt.figure(figsize=(10, 4))
for i in range(vals2.shape[1]):
    plt.plot(grid, vals2[:, i])
plt.title("Drugie pochodne funkcji bazowych")
plt.show()
```

---

# Minimalny import do całego zestawu

```python
import numpy as np
import matplotlib.pyplot as plt

from skfda.representation.basis import (
    ConstantBasis,
    MonomialBasis,
    FourierBasis,
    BSplineBasis,
)
```

---

# Uwaga praktyczna

Jeśli chcesz uruchamiać te przykłady, potrzebujesz biblioteki `scikit-fda`. Dokumentacja oficjalna tej biblioteki zawiera właśnie moduły `ConstantBasis`, `MonomialBasis`, `FourierBasis`, `BSplineBasis` oraz `FDataBasis`. ([fda.readthedocs.io][1])

---

Jeżeli chcesz, mogę teraz zrobić z tego jeszcze **wersję bardziej „notatkową” do Jupyter Markdown**, czyli krótszą i bardziej podobną stylem do Twojego oryginału.

[1]: https://fda.readthedocs.io/en/stable/apilist.html?utm_source=chatgpt.com "API Reference — scikit-fda 0.10.1 documentation"
[2]: https://fda.readthedocs.io/en/stable/modules/representation.html?utm_source=chatgpt.com "Representation of functional Data - scikit-fda's documentation!"
[3]: https://fda.readthedocs.io/en/latest/modules/autosummary/skfda.representation.basis.Fourier.html?utm_source=chatgpt.com "Fourier — scikit-fda 0.10.2.dev0 documentation"
[4]: https://fda.readthedocs.io/en/latest/modules/autosummary/skfda.representation.basis.BSpline.html?utm_source=chatgpt.com "BSpline — scikit-fda 0.10.2.dev0 documentation"
[5]: https://fda.readthedocs.io/en/stable/modules/autosummary/skfda.representation.basis.Basis.html?utm_source=chatgpt.com "Basis — scikit-fda 0.10.1 documentation"
